# Exploratory Data Analysis: Benin - Malanville

This notebook contains comprehensive exploratory data analysis for the Benin solar farm dataset.

## Objectives
1. Summary Statistics & Missing-Value Report
2. Outlier Detection & Basic Cleaning
3. Time Series Analysis
4. Cleaning Impact Analysis
5. Correlation & Relationship Analysis
6. Wind & Distribution Analysis
7. Temperature Analysis
8. Bubble Chart Visualization



In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the data
df = pd.read_csv('../data/benin-malanville.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()



## 1. Summary Statistics & Missing-Value Report



In [ ]:
# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Summary statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("=" * 80)
print("SUMMARY STATISTICS FOR NUMERIC COLUMNS")
print("=" * 80)
print(df[numeric_cols].describe())



In [ ]:
# Missing value analysis
print("=" * 80)
print("MISSING VALUE ANALYSIS")
print("=" * 80)
missing_data = df.isna().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage': missing_percentage
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)
print(missing_df)

# Identify columns with >5% nulls
high_missing = missing_df[missing_df['Percentage'] > 5]
if len(high_missing) > 0:
    print(f"\nColumns with >5% missing values:")
    print(high_missing)
else:
    print("\nNo columns have more than 5% missing values.")



## 2. Outlier Detection & Basic Cleaning

### 2.1 Outlier Detection using Z-scores



In [ ]:
# Key columns for outlier detection
outlier_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']

# Calculate Z-scores
z_scores = {}
outlier_flags = pd.DataFrame(index=df.index)

for col in outlier_cols:
    if col in df.columns:
        z_scores[col] = np.abs(stats.zscore(df[col].dropna()))
        outlier_flags[col] = z_scores[col] > 3
        print(f"{col}: {outlier_flags[col].sum()} outliers (|Z| > 3)")

# Combine outlier flags
outlier_flags['Any_Outlier'] = outlier_flags.any(axis=1)
print(f"\nTotal rows with at least one outlier: {outlier_flags['Any_Outlier'].sum()}")
print(f"Percentage of rows with outliers: {(outlier_flags['Any_Outlier'].sum() / len(df)) * 100:.2f}%")



In [ ]:
# Visualize outliers for key columns
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, col in enumerate(outlier_cols[:7]):
    if col in df.columns:
        # Box plot
        axes[idx].boxplot(df[col].dropna())
        axes[idx].set_title(f'Boxplot: {col}')
        axes[idx].set_ylabel('Value')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



### 2.2 Data Cleaning



In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Handle negative values in solar irradiance (should be >= 0)
solar_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB']
for col in solar_cols:
    if col in df_clean.columns:
        negative_count = (df_clean[col] < 0).sum()
        if negative_count > 0:
            print(f"{col}: {negative_count} negative values found. Replacing with 0.")
            df_clean[col] = df_clean[col].clip(lower=0)

# Handle missing values in key columns using median imputation
key_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'Tamb', 'RH', 'WS']
for col in key_cols:
    if col in df_clean.columns:
        missing_count = df_clean[col].isna().sum()
        if missing_count > 0:
            median_val = df_clean[col].median()
            df_clean[col].fillna(median_val, inplace=True)
            print(f"{col}: Filled {missing_count} missing values with median ({median_val:.2f})")

# Handle outliers - cap extreme values instead of removing (to preserve time series continuity)
for col in outlier_cols:
    if col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 3 * IQR
        upper_bound = Q3 + 3 * IQR
        outliers_count = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
        if outliers_count > 0:
            df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
            print(f"{col}: Capped {outliers_count} outliers using IQR method")

print("\nCleaning completed!")



## 3. Time Series Analysis



In [ ]:
# Set Timestamp as index for time series analysis
df_clean_ts = df_clean.set_index('Timestamp')

# Extract time features
df_clean['Year'] = df_clean['Timestamp'].dt.year
df_clean['Month'] = df_clean['Timestamp'].dt.month
df_clean['Day'] = df_clean['Timestamp'].dt.day
df_clean['Hour'] = df_clean['Timestamp'].dt.hour
df_clean['DayOfWeek'] = df_clean['Timestamp'].dt.dayofweek

# Time series plots for key variables
fig, axes = plt.subplots(4, 1, figsize=(16, 16))

# Sample data for visualization (every 10th point for performance)
sample_idx = df_clean_ts.index[::10]

# GHI
axes[0].plot(sample_idx, df_clean_ts.loc[sample_idx, 'GHI'], alpha=0.6, linewidth=0.5)
axes[0].set_title('Global Horizontal Irradiance (GHI) Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('GHI (W/m²)')
axes[0].grid(True, alpha=0.3)

# DNI
axes[1].plot(sample_idx, df_clean_ts.loc[sample_idx, 'DNI'], alpha=0.6, linewidth=0.5, color='orange')
axes[1].set_title('Direct Normal Irradiance (DNI) Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('DNI (W/m²)')
axes[1].grid(True, alpha=0.3)

# DHI
axes[2].plot(sample_idx, df_clean_ts.loc[sample_idx, 'DHI'], alpha=0.6, linewidth=0.5, color='green')
axes[2].set_title('Diffuse Horizontal Irradiance (DHI) Over Time', fontsize=14, fontweight='bold')
axes[2].set_ylabel('DHI (W/m²)')
axes[2].grid(True, alpha=0.3)

# Ambient Temperature
axes[3].plot(sample_idx, df_clean_ts.loc[sample_idx, 'Tamb'], alpha=0.6, linewidth=0.5, color='red')
axes[3].set_title('Ambient Temperature Over Time', fontsize=14, fontweight='bold')
axes[3].set_ylabel('Temperature (°C)')
axes[3].set_xlabel('Timestamp')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# Monthly patterns
monthly_avg = df_clean.groupby('Month').agg({
    'GHI': 'mean',
    'DNI': 'mean',
    'DHI': 'mean',
    'Tamb': 'mean'
}).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(monthly_avg['Month'], monthly_avg['GHI'], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Average GHI by Month', fontweight='bold')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('GHI (W/m²)')
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].grid(True, alpha=0.3, axis='y')

axes[0, 1].bar(monthly_avg['Month'], monthly_avg['DNI'], color='orange', edgecolor='black')
axes[0, 1].set_title('Average DNI by Month', fontweight='bold')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('DNI (W/m²)')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].grid(True, alpha=0.3, axis='y')

axes[1, 0].bar(monthly_avg['Month'], monthly_avg['DHI'], color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Average DHI by Month', fontweight='bold')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('DHI (W/m²)')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].grid(True, alpha=0.3, axis='y')

axes[1, 1].bar(monthly_avg['Month'], monthly_avg['Tamb'], color='coral', edgecolor='black')
axes[1, 1].set_title('Average Ambient Temperature by Month', fontweight='bold')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Temperature (°C)')
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()



In [ ]:
# Daily patterns (hourly averages)
hourly_avg = df_clean.groupby('Hour').agg({
    'GHI': 'mean',
    'DNI': 'mean',
    'DHI': 'mean',
    'Tamb': 'mean'
}).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(hourly_avg['Hour'], hourly_avg['GHI'], marker='o', linewidth=2, markersize=6, color='skyblue')
axes[0, 0].set_title('Average GHI by Hour of Day', fontweight='bold')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('GHI (W/m²)')
axes[0, 0].set_xticks(range(0, 24, 2))
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(hourly_avg['Hour'], hourly_avg['DNI'], marker='o', linewidth=2, markersize=6, color='orange')
axes[0, 1].set_title('Average DNI by Hour of Day', fontweight='bold')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('DNI (W/m²)')
axes[0, 1].set_xticks(range(0, 24, 2))
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(hourly_avg['Hour'], hourly_avg['DHI'], marker='o', linewidth=2, markersize=6, color='lightgreen')
axes[1, 0].set_title('Average DHI by Hour of Day', fontweight='bold')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('DHI (W/m²)')
axes[1, 0].set_xticks(range(0, 24, 2))
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(hourly_avg['Hour'], hourly_avg['Tamb'], marker='o', linewidth=2, markersize=6, color='coral')
axes[1, 1].set_title('Average Ambient Temperature by Hour of Day', fontweight='bold')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Temperature (°C)')
axes[1, 1].set_xticks(range(0, 24, 2))
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



## 4. Cleaning Impact Analysis



In [ ]:
# Analyze cleaning events impact
if 'Cleaning' in df_clean.columns:
    cleaning_analysis = df_clean.groupby('Cleaning').agg({
        'ModA': 'mean',
        'ModB': 'mean'
    }).reset_index()
    
    cleaning_analysis['Cleaning_Status'] = cleaning_analysis['Cleaning'].map({0: 'No Cleaning', 1: 'After Cleaning'})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x_pos = np.arange(len(cleaning_analysis))
    width = 0.35
    
    axes[0].bar(x_pos, cleaning_analysis['ModA'], width, label='ModA', color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Cleaning Status')
    axes[0].set_ylabel('Average ModA (W/m²)')
    axes[0].set_title('Average ModA Before/After Cleaning', fontweight='bold')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(cleaning_analysis['Cleaning_Status'])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].bar(x_pos, cleaning_analysis['ModB'], width, label='ModB', color='darkorange', edgecolor='black')
    axes[1].set_xlabel('Cleaning Status')
    axes[1].set_ylabel('Average ModB (W/m²)')
    axes[1].set_title('Average ModB Before/After Cleaning', fontweight='bold')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(cleaning_analysis['Cleaning_Status'])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("Cleaning Impact Summary:")
    print(cleaning_analysis[['Cleaning_Status', 'ModA', 'ModB']])
else:
    print("Cleaning column not found in dataset.")



## 5. Correlation & Relationship Analysis



In [ ]:
# Correlation heatmap
corr_cols = ['GHI', 'DNI', 'DHI', 'TModA', 'TModB', 'Tamb', 'RH', 'WS']
corr_data = df_clean[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap: Solar Radiation and Environmental Variables', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()



In [ ]:
# Scatter plots for relationships
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Sample data for scatter plots (every 50th point for performance)
sample_df = df_clean.iloc[::50]

# WS vs GHI
axes[0, 0].scatter(sample_df['WS'], sample_df['GHI'], alpha=0.5, s=10)
axes[0, 0].set_xlabel('Wind Speed (m/s)')
axes[0, 0].set_ylabel('GHI (W/m²)')
axes[0, 0].set_title('Wind Speed vs GHI', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# WSgust vs GHI
axes[0, 1].scatter(sample_df['WSgust'], sample_df['GHI'], alpha=0.5, s=10, color='orange')
axes[0, 1].set_xlabel('Wind Gust Speed (m/s)')
axes[0, 1].set_ylabel('GHI (W/m²)')
axes[0, 1].set_title('Wind Gust Speed vs GHI', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# WD vs GHI
axes[0, 2].scatter(sample_df['WD'], sample_df['GHI'], alpha=0.5, s=10, color='green')
axes[0, 2].set_xlabel('Wind Direction (°N)')
axes[0, 2].set_ylabel('GHI (W/m²)')
axes[0, 2].set_title('Wind Direction vs GHI', fontweight='bold')
axes[0, 2].grid(True, alpha=0.3)

# RH vs Tamb
axes[1, 0].scatter(sample_df['RH'], sample_df['Tamb'], alpha=0.5, s=10, color='red')
axes[1, 0].set_xlabel('Relative Humidity (%)')
axes[1, 0].set_ylabel('Ambient Temperature (°C)')
axes[1, 0].set_title('Relative Humidity vs Ambient Temperature', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# RH vs GHI
axes[1, 1].scatter(sample_df['RH'], sample_df['GHI'], alpha=0.5, s=10, color='purple')
axes[1, 1].set_xlabel('Relative Humidity (%)')
axes[1, 1].set_ylabel('GHI (W/m²)')
axes[1, 1].set_title('Relative Humidity vs GHI', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Tamb vs GHI
axes[1, 2].scatter(sample_df['Tamb'], sample_df['GHI'], alpha=0.5, s=10, color='brown')
axes[1, 2].set_xlabel('Ambient Temperature (°C)')
axes[1, 2].set_ylabel('GHI (W/m²)')
axes[1, 2].set_title('Ambient Temperature vs GHI', fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



## 6. Wind & Distribution Analysis



In [ ]:
# Wind Rose plot (simplified version using bar chart)
if 'WD' in df_clean.columns and 'WS' in df_clean.columns:
    # Create wind direction bins
    df_clean['WD_bin'] = pd.cut(df_clean['WD'], bins=16, labels=range(16))
    wind_rose = df_clean.groupby('WD_bin')['WS'].mean().reset_index()
    wind_rose['Direction'] = wind_rose['WD_bin'].apply(lambda x: x * 22.5)  # Convert to degrees
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    theta = np.radians(wind_rose['Direction'])
    radii = wind_rose['WS']
    bars = ax.bar(theta, radii, width=np.radians(22.5), color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title('Wind Rose: Average Wind Speed by Direction', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Wind Direction', labelpad=20)
    plt.show()
else:
    print("Wind direction or wind speed data not available.")



In [ ]:
# Histograms for GHI and WS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['GHI'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('GHI (W/m²)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Global Horizontal Irradiance (GHI)', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].axvline(df_clean['GHI'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_clean["GHI"].mean():.2f}')
axes[0].legend()

axes[1].hist(df_clean['WS'], bins=50, color='orange', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Wind Speed (m/s)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Wind Speed (WS)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axvline(df_clean['WS'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_clean["WS"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()



## 7. Temperature Analysis



In [ ]:
# Temperature analysis: RH influence on temperature and solar radiation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RH vs Tamb with color coding by GHI
scatter1 = axes[0].scatter(df_clean['RH'], df_clean['Tamb'], c=df_clean['GHI'], 
                          cmap='viridis', alpha=0.6, s=10)
axes[0].set_xlabel('Relative Humidity (%)')
axes[0].set_ylabel('Ambient Temperature (°C)')
axes[0].set_title('RH vs Ambient Temperature (colored by GHI)', fontweight='bold')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='GHI (W/m²)')

# RH vs GHI
axes[1].scatter(df_clean['RH'], df_clean['GHI'], alpha=0.5, s=10, color='purple')
axes[1].set_xlabel('Relative Humidity (%)')
axes[1].set_ylabel('GHI (W/m²)')
axes[1].set_title('Relative Humidity vs GHI', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(df_clean['RH'].dropna(), df_clean['GHI'].dropna(), 1)
p = np.poly1d(z)
axes[1].plot(df_clean['RH'], p(df_clean['RH']), "r--", alpha=0.8, linewidth=2, label='Trend line')
axes[1].legend()

plt.tight_layout()
plt.show()

# Correlation analysis
print("Correlation between RH and Tamb:", df_clean['RH'].corr(df_clean['Tamb']))
print("Correlation between RH and GHI:", df_clean['RH'].corr(df_clean['GHI']))



## 8. Bubble Chart



In [ ]:
# Bubble chart: GHI vs Tamb with bubble size = RH
sample_df = df_clean.iloc[::100]  # Sample for better visualization

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(sample_df['Tamb'], sample_df['GHI'], 
                    s=sample_df['RH']*2,  # Scale RH for bubble size
                    c=sample_df['BP'],  # Color by Barometric Pressure
                    alpha=0.6, cmap='coolwarm', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Ambient Temperature (°C)', fontsize=12)
ax.set_ylabel('GHI (W/m²)', fontsize=12)
ax.set_title('GHI vs Ambient Temperature\n(Bubble size = Relative Humidity, Color = Barometric Pressure)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Barometric Pressure (hPa)', fontsize=10)

# Add legend for bubble size (approximate)
handles = []
for rh_val in [20, 50, 80]:
    handles.append(plt.scatter([], [], s=rh_val*2, c='gray', alpha=0.6, edgecolors='black'))
ax.legend(handles, ['RH: 20%', 'RH: 50%', 'RH: 80%'], 
         scatterpoints=1, loc='upper left', title='Bubble Size Reference')

plt.tight_layout()
plt.show()



## 9. Export Cleaned Data



In [ ]:
# Remove temporary columns added for analysis
export_cols = [col for col in df_clean.columns if col not in ['Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'WD_bin']]
df_export = df_clean[export_cols].copy()

# Export to CSV
output_path = '../data/benin_clean.csv'
df_export.to_csv(output_path, index=False)
print(f"Cleaned data exported to {output_path}")
print(f"Exported shape: {df_export.shape}")
print(f"Columns exported: {len(export_cols)}")



## Summary of Key Findings

### Data Quality
- **Missing Values**: [Summary of missing value patterns]
- **Outliers**: [Summary of outlier detection results]
- **Data Cleaning**: Negative values in solar irradiance columns were corrected, missing values imputed, and outliers capped.

### Key Insights
1. **Solar Irradiance Patterns**: [Key observations about GHI, DNI, DHI patterns]
2. **Seasonal Variations**: [Observations about monthly patterns]
3. **Daily Patterns**: [Observations about hourly patterns]
4. **Environmental Relationships**: [Key correlations and relationships]
5. **Cleaning Impact**: [Impact of cleaning events on module performance]

### Recommendations
- [Actionable insights based on the analysis]

